<a href="https://colab.research.google.com/github/Anuoluwapo65/pytorch-jax-implementation/blob/main/wandb_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import os
import sys
import json
import time
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import CIFAR100
from torchvision import transforms

import tqdm


In [2]:
config = dict(
    num_classes = 100,
    epochs = 156,
    architecture = "CNN",
    batch_size = 128,
    learning_rate = 0.001,
    kernel = [16, 32],
    datasets = "CIFAR100"

)

In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abidoyeanuoluwapo47 (abidoyeanuoluwapo47-obafemi-awolowo-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
def model_pipeline(hyperparameters):
   with wandb.init(project = "CIFAR100", config = hyperparameters):
        config = wandb.config

        model, train_loader, val_loader, test_loader, criterion, optimizer =   make(config, device)
        print(model)
        train_model(model, train_loader, val_loader, criterion, optimizer, config)
        test_model(model, test_loader)
   return model

In [5]:
import torch.utils.data as DataLoader

In [6]:
def set_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

print("using this device", device)

using this device cuda:0


In [7]:
def make(config, device):
  train_set,val_set, test_set = get_data()
  train_loader,val_loader, test_loader  = make_loader(train_set, val_set, test_set, batch_size = config.batch_size)
  model = ConvNet(config.num_classes, config.kernel).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr = config.learning_rate)
  return model, train_loader, val_loader, test_loader, criterion, optimizer

In [8]:
train_datasets = CIFAR100(root = "data_path", download = True)
train_datasets

100%|██████████| 169M/169M [00:26<00:00, 6.50MB/s]


Dataset CIFAR100
    Number of datapoints: 50000
    Root location: data_path
    Split: Train

In [9]:
img_mean = ((train_datasets.data)/255.0).mean(axis = (0,1,2))
img_std = ((train_datasets.data)/255.0).std(axis = (0,1,2))

In [10]:
train_transforms = transforms.Compose([transforms.ToTensor(),
                                       transforms.Normalize(img_mean, img_std),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.RandomResizedCrop(size =32, scale = (0.2, 0.8), ratio = (0.2, 0.8))])
test_transforms = transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize(img_mean, img_std)])

In [11]:
from torchvision.datasets import CIFAR100
from torch.utils.data import DataLoader, random_split
import torch

def get_data():
    # Load dataset ONCE
    full_dataset = CIFAR100(
        root="./data",
        train=True,
        download=True,
        transform=train_transforms
    )

    # Compute split sizes
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    # Reproducible split
    generator = torch.Generator().manual_seed(42)

    train_set, val_set = random_split(
        full_dataset,
        [train_size, val_size],
        generator=generator
    )

    # Test dataset
    test_set = CIFAR100(
        root="./data",
        train=False,
        download=True,
        transform=test_transforms
    )

    return train_set, val_set, test_set


def make_loader(train_set, val_set, test_set, batch_size):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [12]:
# Definition for ConvNet class
class ConvNet(nn.Module):
    def __init__(self, num_classes, kernel_sizes):
        super(ConvNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, kernel_sizes[0], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(kernel_sizes[0], kernel_sizes[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Calculate input features for the fully connected layer
        # Assuming input size is 32x32 after two pooling layers (32/2/2 = 8)
        self.fc = nn.Linear(kernel_sizes[1] * 8 * 8, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        return out

In [13]:
# Definition for train_model function
def train_model(model, train_loader, val_loader, criterion, optimizer, config):
    model.train()
    print(f"\nStarting training for {config.epochs} epochs...")
    for epoch in tqdm.tqdm(range(config.epochs), desc="Epochs"):
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        # Basic validation loop
        model.eval()
        val_loss = 0
        correct = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                val_loss += criterion(output, target).item() * data.size(0)
                pred = output.argmax(dim=1, keepdim=True) # get the index of the max log-probability
                correct += pred.eq(target.view_as(pred)).sum().item()
        val_loss /= len(val_loader.dataset)
        accuracy = 100. * correct / len(val_loader.dataset)
        print(f'Epoch {epoch+1}: Train Loss: {loss.item():.4f}, Val Loss: {val_loss:.4f}, Val Acc: {accuracy:.2f}%')
        model.train() # Set back to train mode
    print("Training finished.")

In [14]:
# Definition for test_model function
def test_model(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item() # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True) # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')

In [15]:
model = model_pipeline(config)
model

100%|██████████| 169M/169M [00:33<00:00, 5.05MB/s]


ConvNet(
  (layer1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=2048, out_features=100, bias=True)
)

Starting training for 156 epochs...


Epochs:   0%|          | 0/156 [00:00<?, ?it/s]

Epoch 1: Train Loss: 3.4821, Val Loss: 3.6762, Val Acc: 15.37%
Epoch 2: Train Loss: 3.6024, Val Loss: 3.5704, Val Acc: 16.74%
Epoch 3: Train Loss: 3.7164, Val Loss: 3.4792, Val Acc: 18.66%
Epoch 4: Train Loss: 3.4001, Val Loss: 3.3912, Val Acc: 21.03%
Epoch 5: Train Loss: 3.0337, Val Loss: 3.3530, Val Acc: 21.40%
Epoch 6: Train Loss: 3.2760, Val Loss: 3.3159, Val Acc: 22.26%
Epoch 7: Train Loss: 3.3276, Val Loss: 3.2591, Val Acc: 22.98%
Epoch 8: Train Loss: 3.1919, Val Loss: 3.2497, Val Acc: 23.65%
Epoch 9: Train Loss: 3.2416, Val Loss: 3.2158, Val Acc: 24.64%
Epoch 10: Train Loss: 3.2180, Val Loss: 3.2215, Val Acc: 23.64%
Epoch 11: Train Loss: 3.0122, Val Loss: 3.2175, Val Acc: 24.53%
Epoch 12: Train Loss: 2.8891, Val Loss: 3.1862, Val Acc: 24.57%
Epoch 13: Train Loss: 3.0588, Val Loss: 3.1808, Val Acc: 25.12%
Epoch 14: Train Loss: 3.0765, Val Loss: 3.1660, Val Acc: 25.35%
Epoch 15: Train Loss: 3.2269, Val Loss: 3.1488, Val Acc: 25.00%
Epoch 16: Train Loss: 3.0455, Val Loss: 3.1371, V

ConvNet(
  (layer1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=2048, out_features=100, bias=True)
)